# M4c-4C — Métricas do modelo (4 cabeças) no conjunto de teste holdout

Variante de `notebooks/bert/10.extrair_metricas_holdout.ipynb` para o modelo com encoder compartilhado + 4 cabeças de classificação (`notebooks/bert_4_cabecas/06.treinar_bertimbau_4_cabecas.ipynb`). Reaproveita o MESMO gabarito de holdout (notebooks 07/09 em `notebooks/bert/`) -- a única coisa que muda é como o modelo é carregado e como a inferência é decodificada.

**Diferença importante em relação ao notebook de referência:** a seção de matriz de confusão a nível de token foi removida -- ela dependia de um único vetor de 9 classes competindo no mesmo softmax, que não existe mais aqui (cada cabeça decide sozinha, em 3 classes, se aquele token é `O`/`B`/`I` da sua própria entidade). O diagnóstico principal passa a ser só a classificação por entidade (seção 2), que é agnóstica à arquitetura.

**Pré-requisitos:** notebooks 07/08/09 (em `notebooks/bert/`) já rodados para o holdout; `notebooks/bert_4_cabecas/06...ipynb` já treinou o modelo.

**Depois deste notebook:** `notebooks/bert_4_cabecas/11.verificar_resultados_manualmente_4_cabecas.ipynb`.

In [ ]:
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print(f"CUDA disponível: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Configuração

`MODELO_GABARITO` deve bater com o que foi usado no notebook 09 (`notebooks/bert/`). `INCLUIR_SINTETICOS`/`CENARIO_MANUAL` escolhem o experimento/cenário treinado, igual `notebooks/bert_4_cabecas/06`.

In [ ]:
def encontrar_raiz_projeto(marcador: str = "requirements.txt") -> Path:
    p = Path.cwd()
    for _ in range(6):
        if (p / marcador).exists():
            return p
        p = p.parent
    raise RuntimeError(f"Não encontrei a raiz do projeto ({marcador}) subindo a partir de {Path.cwd()}")

RAIZ = encontrar_raiz_projeto()

MODELO_GABARITO = "gemini-2.5-flash"
INCLUIR_SINTETICOS = False
CENARIO_MANUAL = None  # ex: "c08_warmup" -- None usa o melhor por F1 do experimento

TIPO_FOCO = "NOME"
N_AMOSTRAS_POR_CATEGORIA = 15
IOU_MINIMO_MATCH = 0.30
MAX_LENGTH = 512
LIMIAR_ALINHAMENTO_MINIMO_PCT = 80

PASTA_EXTRACAO_HOLDOUT = RAIZ / "data" / "documento_social" / "teste_holdout" / "extracao"
PASTA_GABARITO_HOLDOUT = RAIZ / "data" / "documento_social" / "teste_holdout" / "gabarito_alinhado" / MODELO_GABARITO
PASTA_ARQUIVOS_HOLDOUT = RAIZ / "data" / "documento_social" / "teste_holdout" / "arquivos"

TIPOS = ["CPF", "RG", "NOME", "ENDERECO"]
LABEL2ID_POR_TIPO = {tipo: {"O": 0, f"B-{tipo}": 1, f"I-{tipo}": 2} for tipo in TIPOS}
ID2LABEL_POR_TIPO = {tipo: {v: k for k, v in LABEL2ID_POR_TIPO[tipo].items()} for tipo in TIPOS}

_sufixo_experimento = "-com-sinteticos" if INCLUIR_SINTETICOS else "-sem-sinteticos"
PASTA_CENARIOS = RAIZ / "models" / f"bertimbau-4cabecas-pii-v1-cenarios{_sufixo_experimento}"
PASTA_SAIDA_METRICAS = PASTA_CENARIOS / "metricas_holdout"
PASTA_SAIDA_METRICAS.mkdir(parents=True, exist_ok=True)

resultados_cenarios = json.loads((PASTA_CENARIOS / "_comparacao_cenarios.json").read_text(encoding="utf-8"))
cenario_escolhido = next((r for r in resultados_cenarios if r["nome"] == CENARIO_MANUAL), resultados_cenarios[0]) if CENARIO_MANUAL else resultados_cenarios[0]

print(f"Experimento: {PASTA_CENARIOS.name}")
print(f"Cenário avaliado: {cenario_escolhido['nome']} (F1 token-level={cenario_escolhido['eval_f1']:.4f})")
print(f"Pasta do modelo: {cenario_escolhido['pasta_modelo']}")
print(f"Tipo em foco para amostras detalhadas: {TIPO_FOCO}")

## Carrega o gabarito alinhado + extração do holdout

Idêntico ao notebook de referência -- não depende da arquitetura do modelo.

In [ ]:
def bbox_contem_centro(caixa: list, bbox_token: list) -> bool:
    x0, y0, x1, y1 = caixa
    cx = (bbox_token[0] + bbox_token[2]) / 2
    cy = (bbox_token[1] + bbox_token[3]) / 2
    return (x0 - 1) <= cx <= (x1 + 1) and (y0 - 1) <= cy <= (y1 + 1)


def indices_tokens_da_entidade(tokens: list, entidade: dict) -> list:
    caixas = entidade.get("bbox") or []
    indices = []
    for i, tok in enumerate(tokens):
        for caixa in caixas:
            if bbox_contem_centro(caixa, tok["bbox"]):
                indices.append(i)
                break
    return sorted(set(indices))


def construir_exemplos_documento(extracao: dict, gabarito: dict) -> list:
    entidades_por_pagina = {}
    for e in gabarito["entities"]:
        if e.get("match_type") == "unmatched":
            continue
        entidades_por_pagina.setdefault(e["page"], []).append(e)

    exemplos = []
    for pagina_dados in extracao["pages"]:
        tokens = pagina_dados["tokens"]
        if not tokens:
            continue
        exemplos.append({
            "document_id": extracao["document_id"],
            "page": pagina_dados["page"],
            "tokens": [t["text"] for t in tokens],
            "tokens_bbox": [t["bbox"] for t in tokens],
        })
    return exemplos


arquivos_gabarito = sorted(f for f in PASTA_GABARITO_HOLDOUT.glob("*.json") if f.name != "_nao_alinhadas.json")
if not arquivos_gabarito:
    raise SystemExit(f"Nenhum gabarito alinhado encontrado em {PASTA_GABARITO_HOLDOUT} -- rode os notebooks 07 e 09 (em notebooks/bert/) antes.")

exemplos_holdout = []
gabarito_por_doc_pagina = {}
for arq_gab in arquivos_gabarito:
    arq_ext = PASTA_EXTRACAO_HOLDOUT / arq_gab.name
    if not arq_ext.exists():
        print(f"[AVISO] Sem extração local correspondente para {arq_gab.name}, pulando.")
        continue
    gabarito = json.loads(arq_gab.read_text(encoding="utf-8"))
    entidades = gabarito.get("entities", [])
    n_unmatched = sum(1 for e in entidades if e.get("match_type") == "unmatched")
    pct_alinhado = 100 * (len(entidades) - n_unmatched) / len(entidades) if entidades else 100
    if entidades and pct_alinhado < LIMIAR_ALINHAMENTO_MINIMO_PCT:
        print(f"[AVISO] {arq_gab.name}: só {pct_alinhado:.0f}% alinhado, abaixo do limiar ({LIMIAR_ALINHAMENTO_MINIMO_PCT}%) -- pulando.")
        continue
    extracao = json.loads(arq_ext.read_text(encoding="utf-8"))
    exemplos_holdout.extend(construir_exemplos_documento(extracao, gabarito))
    for e in entidades:
        if e.get("match_type") == "unmatched":
            continue
        gabarito_por_doc_pagina.setdefault((gabarito["document_id"], e["page"]), []).append(e)

print(f"Documentos de holdout com gabarito utilizável: {len({ex['document_id'] for ex in exemplos_holdout})} | páginas: {len(exemplos_holdout)}")

## Carrega o modelo (encoder + 4 cabeças) e roda a inferência em todas as páginas do holdout

In [ ]:
class BERTimbau4Cabecas(nn.Module):
    """Mesma definição de notebooks/bert_4_cabecas/06 -- reproduzida aqui por autocontenção
    (cada notebook do projeto é executável isoladamente, sem importar outro notebook)."""

    def __init__(self, checkpoint_base: str, tipos: list):
        super().__init__()
        self.tipos = list(tipos)
        self.encoder = AutoModel.from_pretrained(checkpoint_base)
        hidden = self.encoder.config.hidden_size
        p_dropout = getattr(self.encoder.config, "hidden_dropout_prob", 0.1)
        self.dropout = nn.Dropout(p_dropout)
        self.cabecas = nn.ModuleDict({tipo: nn.Linear(hidden, 3) for tipo in self.tipos})

    def forward(self, input_ids, attention_mask, labels_por_tipo: dict = None):
        saida_encoder = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        oculto = self.dropout(saida_encoder.last_hidden_state)
        logits_por_tipo = {tipo: self.cabecas[tipo](oculto) for tipo in self.tipos}
        return {"logits_por_tipo": logits_por_tipo}

    @classmethod
    def load(cls, pasta, device="cpu"):
        pasta = Path(pasta)
        config = json.loads((pasta / "config_4cabecas.json").read_text(encoding="utf-8"))
        modelo = cls(str(pasta / "encoder"), config["tipos"])
        cabecas_state = torch.load(pasta / "cabecas.pt", map_location=device)
        for tipo in config["tipos"]:
            modelo.cabecas[tipo].load_state_dict(cabecas_state[tipo])
        return modelo.to(device)


tokenizer = AutoTokenizer.from_pretrained(Path(cenario_escolhido["pasta_modelo"]) / "encoder")
model = BERTimbau4Cabecas.load(cenario_escolhido["pasta_modelo"], device=device)
model.eval()
print(f"Modelo carregado: {cenario_escolhido['nome']}")


def agrupar_bboxes_por_linha(tokens_bbox: list, indices: list) -> list:
    if not indices:
        return []
    grupos = []
    grupo_atual = [tokens_bbox[indices[0]]]
    y_ref = tokens_bbox[indices[0]][1]
    for i in indices[1:]:
        bbox = tokens_bbox[i]
        altura_ref = grupo_atual[-1][3] - grupo_atual[-1][1]
        if abs(bbox[1] - y_ref) > max(altura_ref * 0.6, 3):
            grupos.append(grupo_atual)
            grupo_atual = [bbox]
            y_ref = bbox[1]
        else:
            grupo_atual.append(bbox)
    grupos.append(grupo_atual)
    return [[min(b[0] for b in g), min(b[1] for b in g), max(b[2] for b in g), max(b[3] for b in g)] for g in grupos]


from pos_processamento_ner import mesclar_previsoes_por_janela


@torch.no_grad()
def _prever_tags_janela_tipo(tokens_janela: list, tipo: str) -> list:
    encoding = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoding = {k: v.to(device) for k, v in encoding.items()}
    saida = model(**encoding)
    pred_ids = saida["logits_por_tipo"][tipo][0].argmax(dim=-1).tolist()
    word_ids = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH).word_ids()
    tags_por_palavra = ["O"] * len(tokens_janela)
    visto = set()
    for pid, word_idx in zip(pred_ids, word_ids):
        if word_idx is None or word_idx in visto:
            continue
        visto.add(word_idx)
        tags_por_palavra[word_idx] = ID2LABEL_POR_TIPO[tipo][pid]
    return tags_por_palavra


def prever_pagina_4cabecas(tokens: list, tokens_bbox: list) -> list:
    """Prevê a página inteira, uma cabeça (tipo) por vez, dividindo em janelas deslizantes
    com sobreposição quando ultrapassa MAX_LENGTH subtokens (achado 2026-09-15, caso MINAS
    CAL/segundo endereço truncado -- ver scripts/pos_processamento_ner.py). Cada cabeça tem
    sua própria sequência de tags, então a mescla por janela roda uma vez por tipo."""
    entidades = []
    for tipo in TIPOS:
        tags_por_palavra = mesclar_previsoes_por_janela(
            tokens, tokenizer, lambda tj, t=tipo: _prever_tags_janela_tipo(tj, t), max_length=MAX_LENGTH
        )
        i, n = 0, len(tags_por_palavra)
        while i < n:
            if tags_por_palavra[i] == "O":
                i += 1
                continue
            indices = [i]
            j = i + 1
            while j < n and tags_por_palavra[j] == f"I-{tipo}":
                indices.append(j)
                j += 1
            entidades.append({
                "type": tipo,
                "bbox": agrupar_bboxes_por_linha(tokens_bbox, indices),
                "indices": indices,
                "text": " ".join(tokens[k] for k in indices),
            })
            i = j
    return entidades


def bbox_externo(lista_bboxes: list):
    if not lista_bboxes:
        return None
    return [min(b[0] for b in lista_bboxes), min(b[1] for b in lista_bboxes),
            max(b[2] for b in lista_bboxes), max(b[3] for b in lista_bboxes)]


def iou(a, b) -> float:
    if a is None or b is None:
        return 0.0
    xi0, yi0 = max(a[0], b[0]), max(a[1], b[1])
    xi1, yi1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xi1 - xi0) * max(0, yi1 - yi0)
    if inter == 0:
        return 0.0
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    uniao = area_a + area_b - inter
    return inter / uniao if uniao > 0 else 0.0


def contexto_em_torno(tokens: list, indices: list, janela: int = 6) -> tuple:
    ini = max(0, min(indices) - janela) if indices else 0
    fim = min(len(tokens), max(indices) + janela + 1) if indices else min(len(tokens), janela)
    antes = " ".join(tokens[ini:min(indices) if indices else ini])
    depois = " ".join(tokens[(max(indices) + 1) if indices else fim:fim])
    return antes, depois


predicoes_por_pagina = {}
for ex in exemplos_holdout:
    entidades_previstas = prever_pagina_4cabecas(ex["tokens"], ex["tokens_bbox"])
    predicoes_por_pagina[(ex["document_id"], ex["page"])] = {"entidades_previstas": entidades_previstas}

print(f"Inferência concluída em {len(predicoes_por_pagina)} páginas.")

## 1. Classificação detalhada de cada erro (nível de entidade)

(A matriz de confusão a nível de token do notebook de referência não se aplica aqui -- não há um único vetor de classes compartilhado entre as 4 cabeças, ver introdução.)

Para cada entidade do gabarito, classifica o resultado em:
- **TP**: encontrada com IoU >= limiar e tipo certo.
- **FN_perdida**: nenhuma previsão do mesmo tipo sobrepõe (o modelo simplesmente não viu nada ali).
- **FN_tipo_errado**: existe previsão sobrepondo a mesma região, mas de outro tipo.
- **FN_boundary**: existe previsão do tipo certo sobrepondo, mas IoU abaixo do limiar.

Para cada previsão do modelo não usada por nenhum gold:
- **FP_espurio**: não há nenhuma entidade gold na região.

In [ ]:
erros = []

for ex in exemplos_holdout:
    doc_id, pagina = ex["document_id"], ex["page"]
    previstas = predicoes_por_pagina[(doc_id, pagina)]["entidades_previstas"]
    gold = gabarito_por_doc_pagina.get((doc_id, pagina), [])

    usadas_previstas = set()
    for g in gold:
        tipo_gold = g["type"]
        bbox_gold = bbox_externo(g.get("bbox") or [])

        melhor_iou_mesmo_tipo, melhor_idx_mesmo_tipo = 0.0, None
        melhor_iou_qualquer, melhor_idx_qualquer, melhor_tipo_qualquer = 0.0, None, None
        for i, p in enumerate(previstas):
            if id(p) in usadas_previstas:
                continue
            score = iou(bbox_gold, bbox_externo(p["bbox"]))
            if score > melhor_iou_qualquer:
                melhor_iou_qualquer, melhor_idx_qualquer, melhor_tipo_qualquer = score, i, p["type"]
            if p["type"] == tipo_gold and score > melhor_iou_mesmo_tipo:
                melhor_iou_mesmo_tipo, melhor_idx_mesmo_tipo = score, i

        contexto_antes = g.get("context_before", "")
        contexto_depois = g.get("context_after", "")

        if melhor_idx_mesmo_tipo is not None and melhor_iou_mesmo_tipo >= IOU_MINIMO_MATCH:
            usadas_previstas.add(id(previstas[melhor_idx_mesmo_tipo]))
            categoria = "TP"
        elif melhor_idx_mesmo_tipo is not None and melhor_iou_mesmo_tipo > 0:
            categoria = "FN_boundary"
        elif melhor_idx_qualquer is not None and melhor_iou_qualquer >= IOU_MINIMO_MATCH:
            usadas_previstas.add(id(previstas[melhor_idx_qualquer]))
            categoria = "FN_tipo_errado"
        else:
            categoria = "FN_perdida"

        erros.append({
            "document_id": doc_id, "page": pagina, "categoria": categoria,
            "tipo_gold": tipo_gold, "texto_gold": g["text"],
            "tipo_previsto": (melhor_tipo_qualquer if categoria != "TP" else tipo_gold),
            "texto_previsto": (previstas[melhor_idx_qualquer]["text"] if melhor_idx_qualquer is not None else None),
            "iou": round(max(melhor_iou_mesmo_tipo, melhor_iou_qualquer), 3),
            "contexto_antes": contexto_antes, "contexto_depois": contexto_depois,
        })

    for p in previstas:
        if id(p) in usadas_previstas:
            continue
        antes, depois = contexto_em_torno(ex["tokens"], p["indices"])
        erros.append({
            "document_id": doc_id, "page": pagina, "categoria": "FP_espurio",
            "tipo_gold": None, "texto_gold": None,
            "tipo_previsto": p["type"], "texto_previsto": p["text"],
            "iou": 0.0, "contexto_antes": antes, "contexto_depois": depois,
        })

df_erros = pd.DataFrame(erros)
print(f"Total de itens avaliados (TP + erros): {len(df_erros)}")
df_erros["categoria"].value_counts()

In [ ]:
print("=== Categoria de erro por TIPO ===\n")
tabela = pd.crosstab(df_erros["tipo_gold"].fillna("(sem gold — FP)"), df_erros["categoria"])
cols_ordem = [c for c in ["TP", "FN_boundary", "FN_tipo_errado", "FN_perdida", "FP_espurio"] if c in tabela.columns]
tabela = tabela[cols_ordem]
print(tabela)

print("\n=== FP_espurio por tipo previsto ===")
print(df_erros[df_erros["categoria"] == "FP_espurio"]["tipo_previsto"].value_counts())

## 2. Amostras de erro para leitura manual (tipo em foco: `TIPO_FOCO`)

In [ ]:
for categoria in ["FN_perdida", "FN_boundary", "FN_tipo_errado"]:
    subset = df_erros[(df_erros["categoria"] == categoria) & (df_erros["tipo_gold"] == TIPO_FOCO)]
    print(f"\n{'=' * 90}\n{categoria} — {TIPO_FOCO} ({len(subset)} ocorrência(s), mostrando até {N_AMOSTRAS_POR_CATEGORIA})\n{'=' * 90}")
    for _, row in subset.head(N_AMOSTRAS_POR_CATEGORIA).iterrows():
        print(f"  [{row['document_id']} p{row['page']}] gold=\"{row['texto_gold']}\" (iou={row['iou']})")
        print(f"    contexto: ...{row['contexto_antes']} >>>{row['texto_gold']}<<< {row['contexto_depois']}...")
        if row["texto_previsto"]:
            print(f"    modelo previu ({row['tipo_previsto']}): \"{row['texto_previsto']}\"")
        print()

subset_fp = df_erros[(df_erros["categoria"] == "FP_espurio") & (df_erros["tipo_previsto"] == TIPO_FOCO)]
print(f"\n{'=' * 90}\nFP_espurio — {TIPO_FOCO} ({len(subset_fp)} ocorrência(s), mostrando até {N_AMOSTRAS_POR_CATEGORIA})\n{'=' * 90}")
for _, row in subset_fp.head(N_AMOSTRAS_POR_CATEGORIA).iterrows():
    print(f"  [{row['document_id']} p{row['page']}] modelo previu \"{row['texto_previsto']}\" onde não havia entidade")
    print(f"    contexto: ...{row['contexto_antes']} >>>{row['texto_previsto']}<<< {row['contexto_depois']}...\n")

## 3. Ranking dos documentos com mais erros

In [ ]:
df_erros_reais = df_erros[df_erros["categoria"] != "TP"]
ranking = df_erros_reais.groupby("document_id").size().sort_values(ascending=False)
n_gold_por_doc = df_erros[df_erros["tipo_gold"].notna()].groupby("document_id").size()

print(f"{'Documento':<70}{'Erros':<8}{'Total gold':<12}{'% erro'}")
print("-" * 100)
for doc_id, n_erros in ranking.head(20).items():
    n_gold = n_gold_por_doc.get(doc_id, 0)
    pct = 100 * n_erros / n_gold if n_gold else float("nan")
    print(f"{doc_id[:68]:<70}{n_erros:<8}{n_gold:<12}{pct:.1f}%")

## 4. Salva o relatório completo de erros (CSV)

In [ ]:
caminho_csv = PASTA_SAIDA_METRICAS / f"erros_holdout_{cenario_escolhido['nome']}.csv"
df_erros.to_csv(caminho_csv, index=False, encoding="utf-8-sig")
print(f"Relatório completo salvo em: {caminho_csv}")
print(f"({len(df_erros)} linhas — abra no Excel/Sheets para filtrar por categoria/tipo)")

## 5. Pós-processamento heurístico: regex de assinatura digital + propagação de texto idêntico

Mesmas duas correções de `scripts/pos_processamento_ner.py` usadas no notebook de referência (arquitetura-agnósticas — operam só sobre a lista de entidades previstas, não sobre o modelo em si). Inclui a deduplicação de span físico e a correção de acento/span parcial de 2026-09-15.

In [ ]:
import sys

sys.path.insert(0, str(RAIZ / "scripts"))
from pos_processamento_ner import extrair_nome_assinatura_digital, propagar_entidades_por_texto_identico


def avaliar_com_pos_processamento(exemplos: list, aplicar_regex: bool, aplicar_propagacao: bool) -> dict:
    contadores = {t: {"tp": 0, "fp": 0, "fn": 0} for t in TIPOS}

    exemplos_por_doc = defaultdict(list)
    for ex in exemplos:
        exemplos_por_doc[ex["document_id"]].append(ex)

    for doc_id, exs in exemplos_por_doc.items():
        previstas_por_pagina = {}
        entidades_documento = []
        for ex in exs:
            previstas = [dict(p) for p in predicoes_por_pagina[(doc_id, ex["page"])]["entidades_previstas"]]

            if aplicar_regex:
                extras = extrair_nome_assinatura_digital(ex["tokens"], ex["tokens_bbox"])
                for extra in extras:
                    bbox_extra = bbox_externo(extra["bbox"])
                    ja_coberto = any(
                        p["type"] == "NOME" and iou(bbox_extra, bbox_externo(p["bbox"])) > 0.1
                        for p in previstas
                    )
                    if not ja_coberto:
                        previstas.append(extra)

            for p in previstas:
                p["page"] = ex["page"]
            previstas_por_pagina[ex["page"]] = previstas
            entidades_documento.extend(previstas)

        if aplicar_propagacao:
            paginas_doc = [
                {"page": ex["page"], "tokens": [{"text": t, "bbox": b} for t, b in zip(ex["tokens"], ex["tokens_bbox"])]}
                for ex in exs
            ]
            entidades_documento = propagar_entidades_por_texto_identico(
                entidades_documento, paginas_doc, tipos_propagaveis=tuple(TIPOS)
            )
            previstas_por_pagina = defaultdict(list)
            for p in entidades_documento:
                previstas_por_pagina[p["page"]].append(p)

        for ex in exs:
            previstas = previstas_por_pagina.get(ex["page"], [])
            gold = gabarito_por_doc_pagina.get((doc_id, ex["page"]), [])
            previstas_por_tipo = {}
            for p in previstas:
                previstas_por_tipo.setdefault(p["type"], []).append(p)

            usadas = set()
            for g in gold:
                tipo = g["type"]
                candidatos = previstas_por_tipo.get(tipo, [])
                bbox_gold = bbox_externo(g.get("bbox") or [])
                melhor_iou, melhor_idx = 0.0, None
                for i, p in enumerate(candidatos):
                    if id(p) in usadas:
                        continue
                    score = iou(bbox_gold, bbox_externo(p["bbox"]))
                    if score > melhor_iou:
                        melhor_iou, melhor_idx = score, i
                if melhor_idx is not None and melhor_iou >= IOU_MINIMO_MATCH:
                    usadas.add(id(candidatos[melhor_idx]))
                    contadores[tipo]["tp"] += 1
                else:
                    contadores[tipo]["fn"] += 1

            for tipo, candidatos in previstas_por_tipo.items():
                for p in candidatos:
                    if id(p) not in usadas:
                        contadores.setdefault(tipo, {"tp": 0, "fp": 0, "fn": 0})
                        contadores[tipo]["fp"] += 1
    return contadores


def imprimir_pos_proc(titulo: str, contadores: dict) -> dict:
    print(f"\n=== {titulo} ===")
    print(f"{'Tipo':<10}{'TP':<6}{'FP':<6}{'FN':<6}{'Precision':<12}{'Recall':<10}{'F1':<8}")
    tp_g = fp_g = fn_g = 0
    for tipo in TIPOS:
        c = contadores[tipo]
        tp, fp, fn = c["tp"], c["fp"], c["fn"]
        tp_g += tp; fp_g += fp; fn_g += fn
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        print(f"{tipo:<10}{tp:<6}{fp:<6}{fn:<6}{precision:<12.3f}{recall:<10.3f}{f1:<8.3f}")
    precision_g = tp_g / (tp_g + fp_g) if (tp_g + fp_g) else 0.0
    recall_g = tp_g / (tp_g + fn_g) if (tp_g + fn_g) else 0.0
    f1_g = 2 * precision_g * recall_g / (precision_g + recall_g) if (precision_g + recall_g) else 0.0
    print(f"{'GERAL':<10}{tp_g:<6}{fp_g:<6}{fn_g:<6}{precision_g:<12.3f}{recall_g:<10.3f}{f1_g:<8.3f}")
    return {"tp": tp_g, "fp": fp_g, "fn": fn_g, "precision": precision_g, "recall": recall_g, "f1": f1_g}


r_base = imprimir_pos_proc("BASE (sem pós-processamento)", avaliar_com_pos_processamento(exemplos_holdout, False, False))
r_regex = imprimir_pos_proc("+ regex assinatura digital", avaliar_com_pos_processamento(exemplos_holdout, True, False))
r_regex_prop = imprimir_pos_proc("+ regex + propagação de texto idêntico", avaliar_com_pos_processamento(exemplos_holdout, True, True))

print(f"\nF1 GERAL — base: {r_base['f1']:.3f} | +regex: {r_regex['f1']:.3f} | +regex+propagação: {r_regex_prop['f1']:.3f}")
print("\nCompare com notebooks/bert/10 (mesmo holdout, mesmo pós-processamento) para isolar o efeito da arquitetura.")

## 6. Visualização: página do PDF com as caixas sobrepostas

Verde = TP, vermelho = gold perdido (FN), laranja tracejado = previsão sem gold correspondente (FP). Reflete o modelo puro (sem pós-processamento).

In [ ]:
import fitz  # PyMuPDF

DOC_ID_VISUALIZAR = ranking.index[0] if len(ranking) else None
PAGINA_VISUALIZAR = 1

def localizar_pdf_original(document_id: str) -> Path:
    candidato = PASTA_ARQUIVOS_HOLDOUT / document_id
    if candidato.exists():
        return candidato
    achados = list(PASTA_ARQUIVOS_HOLDOUT.glob(f"{Path(document_id).stem}*"))
    if achados:
        return achados[0]
    raise FileNotFoundError(f"PDF original não encontrado para {document_id} em {PASTA_ARQUIVOS_HOLDOUT}")


if DOC_ID_VISUALIZAR:
    caminho_pdf = localizar_pdf_original(DOC_ID_VISUALIZAR)
    doc = fitz.open(caminho_pdf)
    pagina_pdf = doc[PAGINA_VISUALIZAR - 1]

    previstas = predicoes_por_pagina.get((DOC_ID_VISUALIZAR, PAGINA_VISUALIZAR), {}).get("entidades_previstas", [])
    gold = gabarito_por_doc_pagina.get((DOC_ID_VISUALIZAR, PAGINA_VISUALIZAR), [])

    usadas = set()
    for g in gold:
        bbox_gold = bbox_externo(g.get("bbox") or [])
        melhor_iou, melhor_idx = 0.0, None
        for i, p in enumerate(previstas):
            if id(p) in usadas or p["type"] != g["type"]:
                continue
            score = iou(bbox_gold, bbox_externo(p["bbox"]))
            if score > melhor_iou:
                melhor_iou, melhor_idx = score, i
        if melhor_idx is not None and melhor_iou >= IOU_MINIMO_MATCH:
            usadas.add(id(previstas[melhor_idx]))
            cor = (0, 0.6, 0)
        else:
            cor = (0.8, 0, 0)
        # Desenha CADA retângulo por linha de g["bbox"] separadamente -- usar
        # bbox_externo() (um retângulo único cobrindo todas as linhas) faz o desenho
        # "vazar" para a esquerda quando uma entidade quebra linha e a 2ª linha começa
        # mais à esquerda que a 1ª (achado 2026-09-15, caso MINAS CAL/endereço do
        # Sérgio). bbox_externo() continua sendo usado só para o cálculo de IoU acima.
        for retangulo in (g.get("bbox") or []):
            pagina_pdf.draw_rect(fitz.Rect(*retangulo), color=cor, width=1.5)

    for i, p in enumerate(previstas):
        if id(p) in usadas:
            continue
        for retangulo in (p.get("bbox") or []):
            pagina_pdf.draw_rect(fitz.Rect(*retangulo), color=(1, 0.55, 0), width=1.5, dashes="[3] 0")

    pix = pagina_pdf.get_pixmap(dpi=150)
    img_bytes = pix.tobytes("png")
    doc.close()

    from io import BytesIO
    from PIL import Image
    plt.figure(figsize=(11, 15))
    plt.imshow(Image.open(BytesIO(img_bytes)))
    plt.axis("off")
    plt.title(f"{DOC_ID_VISUALIZAR} — página {PAGINA_VISUALIZAR}\nverde=TP  vermelho=FN(perdido)  laranja tracejado=FP")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum documento com erro encontrado (ranking vazio) — nada para visualizar.")